# Module 3 — Standing Up Two-Tier Neptune

**ATLAS: Aligned Three-Layer Architecture for Semantics**  
FSI (Financial Services Industry) Semantic Layer Workshop on AWS (Amazon Web Services)

---

## What this module teaches

Modules 1 and 2 produced an ontology on disk — Turtle files that define classes,
properties, and FIBO (Financial Industry Business Ontology) bindings. That ontology
is correct but inert. It cannot answer questions until it lives in a **graph database**
where queries can traverse it.

A graph database stores data as **nodes** (things) and **edges** (relationships between
things). Unlike a relational database where you join tables, a graph database lets you
follow relationships directly — "start at this Customer, follow the `producesSignal`
edge to a WealthSignal, follow the `hasScore` edge to a Score." That traversal is
what makes competency questions answerable.

**Amazon Neptune** is AWS's managed graph database service. It supports two query
languages: SPARQL (SPARQL Protocol and RDF Query Language) for RDF (Resource Description
Framework) graphs, and Gremlin/openCypher for property graphs. ATLAS uses SPARQL
because our ontology is written in RDF (the Turtle files from Modules 1 and 2 are
RDF serialised as Turtle syntax).

This module makes the ontology physical. You will:

- Deploy two Amazon Neptune serverless clusters: the LGD (Lexical Graph Database)
  and the SLGD (Semantic Layer Graph Database)
- Understand why the two-tier split exists and what each tier is for
- Load the FIBO-aligned ontology into the SLGD
- Run SPARQL discovery queries that confirm the ontology is queryable
- Verify that the LGD is empty and accessible from the SLGD

## Why two clusters, not one

This is the architectural decision that makes ATLAS defensible for regulated
institutions. The two-tier split exists because:

**Regulated banks cannot afford to have probabilistic and unvalidated data in the
same physical store as the data their applications and reports depend on.**

Think of it like a kitchen: raw ingredients (unwashed, uncut, possibly contaminated)
go on one counter. Prepared food (washed, validated, safe to serve) goes on another.
You never mix them. Health inspectors check that you never mix them.

In ATLAS:

- **LGD (Lexical Graph Database)** — the raw-ingredients counter. It holds:
  - Entity resolution candidates ("these two records might be the same person")
  - CDC (Change Data Capture) streams from operational systems
  - Edges extracted from unstructured text by NLP (Natural Language Processing)
  - Anything that has not yet been validated against SHACL (Shapes Constraint
    Language) shapes
  
  The LGD is fast, lossy, and intentionally not authoritative. Nothing in the LGD
  should ever be used as a compliance input.

- **SLGD (Semantic Layer Graph Database)** — the prepared-food counter. It holds:
  - The curated, FIBO-aligned ontology (what we load in this module)
  - Validated instance data that has passed SHACL shapes
  - Reasoner outputs (inferred edges from OWL, the Web Ontology Language)
  - The graph that the application layer queries
  
  Everything in the SLGD has provenance. Everything has been validated. This is the
  graph a regulator can audit.

**Promotion** from LGD to SLGD is a governed, observable, reproducible action —
exactly the audit posture MRM (Model Risk Management) requires. Module 5 builds
the promotion path; this module builds the infrastructure it runs on.

## What is CloudFormation

AWS CloudFormation is an Infrastructure as Code (IaC) service. You write a YAML file
that describes the AWS resources you want (Neptune clusters, security groups, IAM roles),
and CloudFormation creates them for you in the correct order, handling dependencies
automatically. If something fails, it rolls back everything — you never end up with
half-built infrastructure.

The template for this module is at `infrastructure/atlas-neptune-twotier.yaml`.
You do not need to understand every line of it to proceed, but you should know:
- It creates two Neptune clusters (LGD and SLGD)
- It creates a security group that allows traffic on port 8182 (Neptune's default port)
- It creates an IAM (Identity and Access Management) role that lets Neptune read from
  S3 (Amazon Simple Storage Service) — needed for loading ontology files
- It creates an S3 bucket where we stage the ontology files before loading

## What is SPARQL

SPARQL (pronounced "sparkle") is the query language for RDF graphs. It is to a graph
database what SQL is to a relational database. A SPARQL query describes a pattern of
nodes and edges, and the database returns all matches.

Example — "find all OWL classes in the graph":
```sparql
PREFIX owl: <http://www.w3.org/2002/07/owl#>
SELECT ?class WHERE {
    ?class a owl:Class .
}
```

Reading this:
- `PREFIX owl:` declares a shorthand for the OWL namespace (like an import)
- `SELECT ?class` says "return the value of the variable `?class`"
- `?class a owl:Class .` is a triple pattern: "find any node (`?class`) that has
  type (`a` is shorthand for `rdf:type`) equal to `owl:Class`"

You will write more complex SPARQL in later modules. For now, the discovery queries
in this module are simple patterns that confirm the ontology loaded correctly.

## Prerequisites

- Module 2 deliverables (`ontology/atlas-fibo-alignment.ttl` and `ontology/alignment-gaps.md`)
- An AWS account with permissions to create Neptune clusters, IAM roles, and S3 buckets
- A VPC (Virtual Private Cloud) with at least two subnets in different Availability
  Zones (the CloudFormation template handles this requirement)

## Deliverables

- A running two-cluster Neptune environment (LGD + SLGD)
- The SLGD loaded with the FIBO-aligned ontology (353 triples)
- SPARQL discovery query results confirming the graph is queryable

## Architecture class for this module

**DETERMINISTIC.** Infrastructure deployment via CloudFormation is deterministic:
the same template with the same parameters always produces the same resources.
SPARQL queries against a loaded ontology are deterministic: same graph, same query,
same results.

## Key Terms for This Module

If you are new to graph databases or AWS infrastructure, this reference will help.
Each term is also defined in context the first time it appears in the notebook.

| Term | What It Is |
|------|------------|
| **Amazon Neptune** | AWS's fully managed graph database service. It stores data as nodes (things) and edges (relationships) and supports SPARQL queries for RDF graphs. |
| **RDF (Resource Description Framework)** | A W3C standard for representing data as triples: subject-predicate-object statements. Every triple says "this thing has this relationship to that thing." Our Turtle (.ttl) files are RDF serialised in Turtle syntax. |
| **SPARQL (SPARQL Protocol and RDF Query Language)** | The query language for RDF graphs. Pronounced "sparkle." It describes patterns of nodes and edges, and the database returns all matches. Equivalent to SQL for relational databases. |
| **Turtle (.ttl)** | A human-readable file format for writing RDF triples. The ontology files from Modules 1 and 2 are Turtle files. |
| **Triple** | The atomic unit of data in an RDF graph: one subject, one predicate, one object. Example: `atlas:Customer rdf:type owl:Class .` says "Customer is a type of OWL Class." |
| **LGD (Lexical Graph Database)** | The first of ATLAS's two Neptune clusters. Holds raw, unvalidated data from source systems. Think of it as the staging area before quality checks. |
| **SLGD (Semantic Layer Graph Database)** | The second Neptune cluster. Holds curated, FIBO-aligned, SHACL-validated data. This is the authoritative graph that applications query. |
| **S3 (Amazon Simple Storage Service)** | AWS's object storage service. Files (objects) are stored in buckets. Neptune reads ontology files from S3 during bulk loading. |
| **IAM (Identity and Access Management)** | AWS's permission system. IAM roles define what a service (like Neptune) is allowed to do (like read from a specific S3 bucket). |
| **VPC (Virtual Private Cloud)** | A logically isolated network within AWS. Neptune runs inside a VPC and is only accessible from other resources in the same VPC (or via peering). This is a security feature, not a limitation. |
| **CloudFormation** | AWS's Infrastructure as Code service. You describe resources in a YAML template, and CloudFormation creates, updates, or deletes them as a single unit (called a "stack"). |
| **Security Group** | A virtual firewall that controls which network traffic can reach a resource. Our security group allows traffic on port 8182 (Neptune's port) only from within the VPC. |
| **Availability Zone (AZ)** | A physically separate data centre within an AWS region. Neptune requires subnets in at least two AZs for resilience — if one data centre fails, the other keeps running. |
| **Bulk Loading** | Neptune's optimised path for loading large amounts of RDF data from S3 in a single operation. Faster than inserting triples one at a time via SPARQL. |
| **NCU (Neptune Capacity Unit)** | The unit of compute capacity for Neptune Serverless. More NCUs = more processing power. The workshop uses 1–2.5 NCUs (minimum configuration). |
| **SHACL (Shapes Constraint Language)** | A W3C standard for writing validation rules on graph data. In ATLAS, SHACL shapes enforce the boundary between deterministic and probabilistic data. Covered in depth in Module 6. |
| **OWL (Web Ontology Language)** | A W3C standard for defining ontology classes and their logical relationships. The `atlas-core.ttl` file uses OWL to declare classes and properties. |
| **FIBO (Financial Industry Business Ontology)** | The industry-standard vocabulary for financial services, published by the EDM Council. ATLAS aligns to FIBO via `rdfs:subClassOf` bindings (Module 2). |

## Cell 2 — Deploy the Neptune Infrastructure

### What we are deploying

The CloudFormation template at `infrastructure/atlas-neptune-twotier.yaml` creates
six resources:

| Resource | What It Is | Why We Need It |
|----------|-----------|----------------|
| Neptune cluster `atlas-lgd` | A serverless graph database | The raw-data tier — receives unvalidated triples from source systems |
| Neptune cluster `atlas-slgd` | A serverless graph database | The curated tier — holds the FIBO-aligned, SHACL-validated ontology and instance data |
| Security group | A network firewall rule | Allows traffic on port 8182 (Neptune's port) only from within the VPC |
| Subnet group | A set of network subnets | Tells Neptune which network segments to deploy into (requires 2+ Availability Zones for resilience) |
| IAM role | A permission set | Lets Neptune read files from S3 — needed for loading ontology files |
| S3 bucket | A file storage location | Where we stage ontology files before Neptune loads them |

### What "serverless" means for Neptune

Neptune Serverless automatically scales compute capacity based on workload. You set
a minimum and maximum capacity (measured in NCUs — Neptune Capacity Units), and
Neptune scales between them. For the workshop, we use 1–2.5 NCUs, which is the
smallest possible configuration. This keeps costs low (~$0.35/hour per cluster)
while providing enough capacity for the ontology and synthetic data.

### What happens when you run the next cell

If the CloudFormation stack has already been deployed (you ran the `aws cloudformation
create-stack` command earlier), the cell below retrieves the endpoints. If the stack
is still creating, it waits until Neptune is ready.

Neptune clusters take approximately 5–10 minutes to provision. This is normal —
Neptune is creating database instances, configuring networking, and setting up
replication. The cell will print progress as it waits.

### If you have not yet deployed the stack

Run this command in a terminal first (replacing the VPC and subnet IDs with your own):

```bash
aws cloudformation create-stack \\
  --stack-name atlas-neptune-twotier \\
  --template-body file://infrastructure/atlas-neptune-twotier.yaml \\
  --parameters \\
    ParameterKey=VpcId,ParameterValue=vpc-XXXXXXXXX \\
    ParameterKey=SubnetIds,ParameterValue=subnet-AAA\\,subnet-BBB\\,subnet-CCC \\
  --capabilities CAPABILITY_NAMED_IAM \\
  --region us-east-1
```

The `--capabilities CAPABILITY_NAMED_IAM` flag is required because the template
creates an IAM role. AWS requires you to explicitly acknowledge this for security.

In [ ]:
import sys
sys.path.insert(0, "../notebooks/shared")

import boto3
import time

# AWS CloudFormation client — used to check the stack status and retrieve outputs
cfn = boto3.client('cloudformation', region_name='us-east-1')
STACK_NAME = 'atlas-neptune-twotier'

# Check if the stack exists and wait for it to complete
print(f'Checking CloudFormation stack: {STACK_NAME}')
print('(If the stack is still creating, this cell will wait. Neptune takes 5-10 minutes.)')
print()

try:
    response = cfn.describe_stacks(StackName=STACK_NAME)
    status = response['Stacks'][0]['StackStatus']
    print(f'Current status: {status}')
    
    if status == 'CREATE_IN_PROGRESS':
        print('Waiting for stack to complete...')
        waiter = cfn.get_waiter('stack_create_complete')
        waiter.wait(StackName=STACK_NAME, WaiterConfig={'Delay': 30, 'MaxAttempts': 40})
        response = cfn.describe_stacks(StackName=STACK_NAME)
        status = response['Stacks'][0]['StackStatus']
        print(f'Final status: {status}')
except cfn.exceptions.ClientError as e:
    print(f'Stack not found: {e}')
    print('Deploy the stack first — see the instructions in the cell above.')
    raise

# Extract the outputs — these are the values we need for the rest of the module
stack = response['Stacks'][0]
outputs = {o['OutputKey']: o['OutputValue'] for o in stack.get('Outputs', [])}

# Neptune endpoints — these are the network addresses of the two graph databases
LGD_ENDPOINT = outputs.get('LGDEndpoint', '')
SLGD_ENDPOINT = outputs.get('SLGDEndpoint', '')
LGD_PORT = outputs.get('LGDPort', '8182')
SLGD_PORT = outputs.get('SLGDPort', '8182')

# Supporting resources
S3_BUCKET = outputs.get('OntologyStagingBucketName', '')
NEPTUNE_S3_ROLE = outputs.get('NeptuneS3RoleArn', '')

print()
print('Neptune Endpoints (save these — used in every subsequent module):')
print(f'  LGD  (raw data):     {LGD_ENDPOINT}:{LGD_PORT}')
print(f'  SLGD (curated data): {SLGD_ENDPOINT}:{SLGD_PORT}')
print(f'  S3 staging bucket:   {S3_BUCKET}')
print(f'  Neptune IAM role:    {NEPTUNE_S3_ROLE}')
print()
print('Both clusters are Neptune Serverless (1–2.5 NCU).')
print('Cost: approximately $0.35/hour per cluster while running.')

## Cell 4 — Upload Ontology Files to S3 (Amazon Simple Storage Service)

### Why we stage files in S3 before loading into Neptune

Neptune's bulk loader reads data from S3 — it does not accept file uploads directly.
This is a deliberate design choice by AWS: S3 is the universal staging area for data
in AWS, and Neptune integrates with it via an IAM role (the role we created in the
CloudFormation template).

The flow is:
1. We upload our Turtle (.ttl) files to the S3 bucket
2. We tell Neptune "load everything in this S3 path"
3. Neptune reads the files using its IAM role and parses the triples into the graph

### What files we are loading

We load four files into the SLGD (the curated tier):

| File | Module | What It Contains |
|------|--------|------------------|
| `atlas-core.ttl` | Module 1 | The 18-class starter ontology (classes, properties, constraints) |
| `atlas-fibo-alignment.ttl` | Module 2 | FIBO bindings (`rdfs:subClassOf`) and 3 new classes |
| `skos-codelists.ttl` | Module 1 | The WealthSignalType and RoutingRoute concept schemes |
| `gleif-bindings.ttl` | Module 2 | LEI (Legal Entity Identifier) datatype properties |

The LGD stays empty. It will be populated in Module 4 when we connect source systems.

### What "bulk loading" means

Bulk loading is Neptune's optimised path for loading large amounts of data at once.
It is faster than inserting triples one at a time via SPARQL because Neptune can
batch the parsing and indexing. For our 353 triples it makes little difference, but
for production ontologies with millions of triples, bulk loading is essential.

An alternative approach (used when bulk loading is unavailable) is SPARQL UPDATE
with INSERT DATA statements. The notebook includes both paths.

In [ ]:
import boto3
from pathlib import Path

# S3 client — used to upload files to the staging bucket
s3 = boto3.client('s3', region_name='us-east-1')

# The four ontology files to load into the SLGD
ontology_files = [
    '../ontology/atlas-core.ttl',
    '../ontology/atlas-fibo-alignment.ttl',
    '../ontology/extensions/skos-codelists.ttl',
    '../ontology/extensions/gleif-bindings.ttl',
]

print(f'Uploading ontology files to S3')
print(f'  Bucket: {S3_BUCKET}')
print(f'  Prefix: ontology/')
print()

for filepath in ontology_files:
    p = Path(filepath)
    if not p.exists():
        print(f'  [SKIP] {filepath} — file not found')
        continue
    
    # The S3 key is the path within the bucket where the file will be stored
    key = f'ontology/{p.name}'
    s3.upload_file(str(p), S3_BUCKET, key)
    size_kb = p.stat().st_size / 1024
    print(f'  [OK] {p.name} ({size_kb:.1f} KB) -> s3://{S3_BUCKET}/{key}')

print(f'\nAll ontology files staged in S3. Ready for Neptune bulk load.')

## Cell 6 — Load the Ontology into the SLGD

### What happens during loading

When Neptune loads a Turtle file, it:
1. Reads each triple (subject, predicate, object) from the file
2. Resolves all prefixes to full IRIs (Internationalised Resource Identifiers)
3. Indexes the triples for fast lookup by subject, predicate, and object
4. Makes the triples immediately queryable via SPARQL

After loading, a SPARQL query like `SELECT ?class WHERE { ?class a owl:Class }` will
find all 21 atlas: classes we defined in Modules 1 and 2.

### Two loading approaches

This cell attempts Neptune's bulk loader first (fastest, reads from S3 directly).
If the bulk loader is unavailable (common on cold-start serverless instances), it
falls back to SPARQL INSERT DATA (slower but always works).

Both approaches produce the same result: 353 triples in the SLGD.

In [ ]:
import requests
import json
import time
from rdflib import Graph

SLGD_URL = f'https://{SLGD_ENDPOINT}:{SLGD_PORT}'

def sparql_query(endpoint, port, query):
    """Execute a read-only SPARQL query against a Neptune endpoint.
    
    Neptune exposes SPARQL at https://<endpoint>:<port>/sparql.
    Read queries use the 'query' parameter; write queries use 'update'.
    """
    url = f'https://{endpoint}:{port}/sparql'
    response = requests.post(
        url,
        data={'query': query},
        headers={'Accept': 'application/sparql-results+json'},
        timeout=30
    )
    if response.status_code == 200:
        return response.json()
    else:
        raise Exception(f'SPARQL failed ({response.status_code}): {response.text[:200]}')

def sparql_update(endpoint, port, update_query):
    """Execute a SPARQL UPDATE (INSERT/DELETE) against a Neptune endpoint.
    
    Note the parameter name is 'update', not 'query' — this tells Neptune
    the request modifies data rather than just reading it.
    """
    url = f'https://{endpoint}:{port}/sparql'
    response = requests.post(
        url,
        data={'update': update_query},
        headers={'Content-Type': 'application/x-www-form-urlencoded'},
        timeout=60
    )
    return response

# Check current triple count
result = sparql_query(SLGD_ENDPOINT, SLGD_PORT,
    'SELECT (COUNT(*) AS ?cnt) WHERE { ?s ?p ?o }')
current_count = int(result['results']['bindings'][0]['cnt']['value'])
print(f'SLGD current triple count: {current_count}')

if current_count >= 300:
    print(f'Ontology already loaded ({current_count} triples). Skipping load.')
else:
    print(f'Loading ontology via SPARQL INSERT DATA...')
    print('(This takes 30-60 seconds for 353 triples)')
    print()
    
    # Load all ontology files into rdflib locally
    g = Graph()
    g.parse('../ontology/atlas-core.ttl', format='turtle')
    g.parse('../ontology/atlas-fibo-alignment.ttl', format='turtle')
    g.parse('../ontology/extensions/skos-codelists.ttl', format='turtle')
    g.parse('../ontology/extensions/gleif-bindings.ttl', format='turtle')
    
    # Serialise to N-Triples (one triple per line, unambiguous format)
    ntriples = g.serialize(format='nt')
    lines = [l.strip() for l in ntriples.split('\n') if l.strip() and not l.startswith('#')]
    print(f'  Triples to load: {len(lines)}')
    
    # Send in batches of 30 triples via SPARQL INSERT DATA
    BATCH_SIZE = 30
    loaded = 0
    errors = 0
    
    for i in range(0, len(lines), BATCH_SIZE):
        batch = lines[i:i+BATCH_SIZE]
        insert_query = 'INSERT DATA {\n' + '\n'.join(batch) + '\n}'
        resp = sparql_update(SLGD_ENDPOINT, SLGD_PORT, insert_query)
        
        if resp.status_code == 200:
            loaded += len(batch)
        else:
            errors += 1
            if errors <= 2:
                print(f'  Batch error: {resp.text[:100]}')
        
        if (i // BATCH_SIZE + 1) % 4 == 0:
            print(f'  Progress: {loaded}/{len(lines)} triples')
    
    print(f'\n  Load complete: {loaded} triples loaded, {errors} errors')
    
    # Verify
    result = sparql_query(SLGD_ENDPOINT, SLGD_PORT,
        'SELECT (COUNT(*) AS ?cnt) WHERE { ?s ?p ?o }')
    final_count = int(result['results']['bindings'][0]['cnt']['value'])
    print(f'  SLGD final triple count: {final_count}')

## Cell 8 — SPARQL Discovery Queries

### What discovery queries are and why we run them

A discovery query is a SPARQL query that explores the structure of the graph rather
than answering a business question. Think of it as running `SHOW TABLES` and
`DESCRIBE table` in SQL — you are confirming the schema loaded correctly before
you start querying for business answers.

We run four discovery queries:

1. **List all OWL classes** — confirms the ontology's class hierarchy loaded
2. **List classes with subclasses** — confirms the FIBO alignment bindings are
   visible (e.g., `fibo:IndependentParty` should have `atlas:Customer` as a subclass)
3. **List object properties with domain and range** — confirms the relationships
   between classes are queryable
4. **Count triples in the LGD** — confirms the LGD is empty (it should be —
   we only loaded data into the SLGD)

### Reading the SPARQL results

Each query returns a table of results. The columns are the variables from the
SELECT clause (e.g., `?class`, `?label`). Each row is one match in the graph.

If a query returns zero rows, something went wrong with the load. The most common
cause: the ontology files were not uploaded to S3, or the Neptune IAM role does not
have permission to read from the bucket.

In [ ]:
# Discovery Query 1: All OWL classes in the SLGD
# This confirms the ontology loaded correctly.

q1 = """
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX atlas: <https://github.com/your-org/atlas/ontology#>

SELECT ?class ?label WHERE {
    ?class a owl:Class .
    FILTER(STRSTARTS(STR(?class), STR(atlas:)))
    OPTIONAL { ?class rdfs:label ?label }
}
ORDER BY ?class
"""

print('Discovery Query 1: All atlas: OWL classes in the SLGD')
print('=' * 70)
result1 = sparql_query(SLGD_ENDPOINT, SLGD_PORT, q1)
classes_found = result1['results']['bindings']
print(f'Classes found: {len(classes_found)}')
print()
print(f'{"Class":<30} {"Label"}')
print('-' * 60)
for b in classes_found:
    cls = b['class']['value'].split('#')[-1]
    label = b.get('label', {}).get('value', '(no label)')
    print(f'  atlas:{cls:<28} {label}')

print(f'\nExpected: >= 18 classes (18 from Module 1 + 3 from Module 2)')
print(f'Actual:   {len(classes_found)} classes')
assert len(classes_found) >= 18, f'Expected >= 18 classes, got {len(classes_found)}'

In [ ]:
# Discovery Query 2: Classes with subclasses (shows FIBO alignment)
# This confirms that the rdfs:subClassOf bindings from Module 2 are visible.
# For example, fibo:IndependentParty should show atlas:Customer as a subclass.

q2 = """
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>

SELECT ?parent (COUNT(?child) AS ?subclassCount) WHERE {
    ?child rdfs:subClassOf ?parent .
    ?child a owl:Class .
}
GROUP BY ?parent
ORDER BY DESC(?subclassCount)
"""

print('Discovery Query 2: Classes with subclasses (FIBO alignment visible)')
print('=' * 70)
result2 = sparql_query(SLGD_ENDPOINT, SLGD_PORT, q2)
print(f'{"Parent Class":<65} {"Subclasses"}')
print('-' * 80)
for b in result2['results']['bindings']:
    parent = b['parent']['value']
    # Shorten for display
    short = parent.replace('https://spec.edmcouncil.org/fibo/ontology/', 'fibo:')
    short = short.replace('http://www.w3.org/ns/prov#', 'prov:')
    short = short.replace('http://www.w3.org/ns/dcat#', 'dcat:')
    short = short.replace('http://www.w3.org/2004/02/skos/core#', 'skos:')
    count = b['subclassCount']['value']
    print(f'  {short:<63} {count}')

print()
print('Reading this: each row shows a FIBO or extension-ring class and how many')
print('atlas: classes are bound to it via rdfs:subClassOf. This is the FIBO')
print('alignment from Module 2, now visible as queryable graph structure.')

In [ ]:
# Discovery Query 3: Object properties with domain and range
# This confirms the relationships between classes are queryable.
# Domain = the class at the start of the relationship
# Range = the class at the end of the relationship

q3 = """
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX atlas: <https://github.com/your-org/atlas/ontology#>

SELECT ?prop ?domain ?range WHERE {
    ?prop a owl:ObjectProperty .
    FILTER(STRSTARTS(STR(?prop), STR(atlas:)))
    OPTIONAL { ?prop rdfs:domain ?domain }
    OPTIONAL { ?prop rdfs:range ?range }
}
ORDER BY ?prop
"""

print('Discovery Query 3: Object properties (relationships between classes)')
print('=' * 70)
result3 = sparql_query(SLGD_ENDPOINT, SLGD_PORT, q3)
print(f'Object properties found: {len(result3["results"]["bindings"])}')
print()
print(f'{"Property":<25} {"Domain (from)":<22} {"Range (to)"}')
print('-' * 70)
for b in result3['results']['bindings']:
    prop = b['prop']['value'].split('#')[-1]
    domain = b.get('domain', {}).get('value', '').split('#')[-1] or '(any)'
    rng = b.get('range', {}).get('value', '').split('#')[-1] or '(any)'
    print(f'  {prop:<23} {domain:<20} -> {rng}')

print()
print('Reading this: each row is a relationship you can traverse in SPARQL.')
print('For example, "hasAccount  Customer -> Account" means you can write:')
print('  ?customer atlas:hasAccount ?account .')
print('and SPARQL will find all Customer-to-Account links in the graph.')

In [ ]:
# Discovery Query 4: Confirm the LGD is empty
# The LGD should have zero triples at this point. It will be populated
# in Module 4 when we connect source systems.

print('Discovery Query 4: LGD triple count')
print('=' * 70)

result4 = sparql_query(LGD_ENDPOINT, LGD_PORT,
    'SELECT (COUNT(*) AS ?cnt) WHERE { ?s ?p ?o }')
lgd_count = int(result4['results']['bindings'][0]['cnt']['value'])

print(f'LGD triples: {lgd_count}')
if lgd_count == 0:
    print('[PASS] LGD is empty as expected.')
    print()
    print('The LGD will be populated in Module 4 when we connect:')
    print('  - Pattern A: Customer master data via S3 Iceberg')
    print('  - Pattern B: Transaction history via Snowflake Horizon (or Athena fallback)')
    print('  - Pattern C: Real-time events via Amazon Kinesis / MSK (Managed Streaming for Apache Kafka)')
else:
    print(f'[WARN] LGD has {lgd_count} triples. Expected 0 at this stage.')
    print('If you loaded test data, clear it with: DROP ALL')

## Cell 13 — Module 3 Validation Gate

The validation gate confirms that the two-tier Neptune infrastructure is operational
and the ontology is loaded correctly. All four checks must pass before proceeding
to Module 4.

| Gate | What It Checks | Why It Matters |
|------|---------------|----------------|
| 1 | SLGD has >= 18 atlas: classes | Confirms the ontology loaded into the correct cluster |
| 2 | LGD has 0 triples | Confirms we did not accidentally load into the wrong tier |
| 3 | SLGD is queryable (total triple count > 0) | Confirms SPARQL is operational |
| 4 | Both clusters report `available` status | Confirms infrastructure is healthy |

In [ ]:
import boto3

print('=' * 60)
print('MODULE 3 VALIDATION GATE')
print('=' * 60)
print()

gate_pass = True

# Gate 1: SLGD has atlas: classes
q_classes = """
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX atlas: <https://github.com/your-org/atlas/ontology#>
SELECT (COUNT(?c) AS ?cnt) WHERE {
    ?c a owl:Class .
    FILTER(STRSTARTS(STR(?c), STR(atlas:)))
}
"""
try:
    r = sparql_query(SLGD_ENDPOINT, SLGD_PORT, q_classes)
    atlas_class_count = int(r['results']['bindings'][0]['cnt']['value'])
    if atlas_class_count >= 18:
        print(f'[PASS] Gate 1 \u2014 SLGD has {atlas_class_count} atlas: classes (expected >= 18)')
    else:
        print(f'[FAIL] Gate 1 \u2014 SLGD has {atlas_class_count} atlas: classes (expected >= 18)')
        gate_pass = False
except Exception as e:
    print(f'[FAIL] Gate 1 \u2014 SPARQL query failed: {e}')
    gate_pass = False

# Gate 2: LGD is empty
try:
    r2 = sparql_query(LGD_ENDPOINT, LGD_PORT, 'SELECT (COUNT(*) AS ?cnt) WHERE { ?s ?p ?o }')
    lgd_triples = int(r2['results']['bindings'][0]['cnt']['value'])
    if lgd_triples == 0:
        print(f'[PASS] Gate 2 \u2014 LGD is empty ({lgd_triples} triples)')
    else:
        print(f'[WARN] Gate 2 \u2014 LGD has {lgd_triples} triples (expected 0)')
except Exception as e:
    print(f'[FAIL] Gate 2 \u2014 LGD query failed: {e}')
    gate_pass = False

# Gate 3: SLGD queryable
try:
    r3 = sparql_query(SLGD_ENDPOINT, SLGD_PORT, 'SELECT (COUNT(*) AS ?cnt) WHERE { ?s ?p ?o }')
    total = int(r3['results']['bindings'][0]['cnt']['value'])
    if total > 0:
        print(f'[PASS] Gate 3 \u2014 SLGD queryable ({total} total triples)')
    else:
        print(f'[FAIL] Gate 3 \u2014 SLGD is empty (0 triples)')
        gate_pass = False
except Exception as e:
    print(f'[FAIL] Gate 3 \u2014 SLGD not queryable: {e}')
    gate_pass = False

# Gate 4: Cluster status via AWS management API
neptune = boto3.client('neptune', region_name='us-east-1')
for cluster_id in ['atlas-lgd', 'atlas-slgd']:
    try:
        desc = neptune.describe_db_clusters(DBClusterIdentifier=cluster_id)
        status = desc['DBClusters'][0]['Status']
        if status == 'available':
            print(f'[PASS] Gate 4 \u2014 {cluster_id} status: {status}')
        else:
            print(f'[FAIL] Gate 4 \u2014 {cluster_id} status: {status} (expected: available)')
            gate_pass = False
    except Exception as e:
        print(f'[FAIL] Gate 4 \u2014 {cluster_id}: {e}')
        gate_pass = False

print()
if gate_pass:
    print('MODULE 3 VALIDATION: PASS')
    print('You may proceed to Module 4.')
else:
    print('MODULE 3 VALIDATION: FAIL')
    print('Fix the failing gate(s) above before proceeding to Module 4.')
    raise AssertionError('Module 3 validation gate failed.')

## Extending This to Your Data

### Sizing the two clusters for production

The workshop uses Neptune Serverless with 1–2.5 NCU (Neptune Capacity Units),
which is the smallest possible configuration. For production:

- **LGD**: Size for write throughput. The LGD receives CDC (Change Data Capture)
  streams and entity resolution candidates continuously. Start with `db.r6g.large`
  and scale based on your ingestion rate. Monitor the `GremlinRequestsPerSec` and
  `SparqlRequestsPerSec` CloudWatch metrics.
- **SLGD**: Size for read throughput. The SLGD serves application queries from
  AppSync, Bedrock agents, and reporting tools. Add read replicas when P99 query
  latency exceeds 500ms under normal load.

### When to add read replicas

Add a read replica to the SLGD when:
- P99 (99th percentile) query latency exceeds 500ms under normal load
- You have multiple application consumers (AWS AppSync, Amazon Bedrock agent,
  reporting dashboards) that would benefit from query isolation
- You need to run long-running analytical queries without affecting real-time
  application performance

### The most common networking gotcha

**Neptune SPARQL SERVICE (federated query between clusters) requires both clusters
in the same VPC or a VPC peering with explicit routes.**

If you deploy the LGD and SLGD in different VPCs (common in multi-account setups
where production and development are separated), the SPARQL SERVICE call will time
out silently — no error message, just no results.

The fix: VPC peering with route table entries for the Neptune endpoint CIDR
(Classless Inter-Domain Routing) ranges, or deploy both clusters in the same VPC
with security group rules that allow port 8182 traffic between them.

### Binding to existing IAM roles

The workshop creates its own IAM role (`atlas-neptune-s3-access`). In production,
you likely have existing roles with S3 access. To use them:

1. Add the Neptune service principal (`rds.amazonaws.com`) to the role's trust policy
2. Associate the role with the Neptune cluster:
   ```bash
   aws neptune add-role-to-db-cluster \\
     --db-cluster-identifier atlas-slgd \\
     --role-arn arn:aws:iam::ACCOUNT:role/your-existing-role
   ```
3. Remove the workshop-created role when you are done

## What Changed

Module 3 added the following to the ATLAS architecture:

| Artifact | Location | Description |
|----------|----------|-------------|
| CloudFormation template | `infrastructure/atlas-neptune-twotier.yaml` | Deploys two Neptune Serverless clusters, security group, subnet group, S3 bucket, and IAM role |
| LGD cluster | `atlas-lgd` (Neptune Serverless) | Empty graph database; ready for Module 4 source connections |
| SLGD cluster | `atlas-slgd` (Neptune Serverless) | Loaded with 353 triples: atlas-core.ttl + atlas-fibo-alignment.ttl + SKOS codelists + GLEIF bindings |
| S3 staging bucket | `atlas-ontology-staging-<account-id>` | Ontology files staged for Neptune loading |

**Key architectural point established:**

The two-tier split is now a physical reality, not a diagram. The LGD and SLGD are
separate Neptune clusters with separate endpoints, separate security groups, and
separate IAM roles. Data cannot accidentally flow from one to the other — promotion
from LGD to SLGD requires an explicit, auditable action (built in Module 5).

**What Module 4 builds on this:**

Module 4 connects three source systems to the LGD:
- Pattern A: Customer master data via S3 Iceberg tables and Ontop VKG (Virtual
  Knowledge Graph) with R2RML (Relational-to-RDF Mapping Language) mappings
- Pattern B: Transaction history via Snowflake Horizon (or Amazon Athena fallback)
- Pattern C: Real-time events via Amazon Kinesis / Amazon MSK (Managed Streaming
  for Apache Kafka)

The LGD — currently empty — will be populated with raw triples from all three
sources. The SLGD remains untouched until Module 5's promotion path moves
validated, entity-resolved data from LGD to SLGD with full PROV-O (W3C Provenance
Ontology) attribution.